In [3]:
import pandas as pd

#load data
df = pd.read_csv('Dataset/Dataset1.csv')

df

,timestamp,source,log_message,target_label
0,27-06-2025 07:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert
3,12-07-2025 00:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status
4,02-06-2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status
...,...,...,...,...
2405,13-08-2025 07:29,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status
2406,01-11-2025 05:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert
2407,03-08-2025 03:07,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status
2408,11-11-2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error


In [4]:
df.source.unique()

<StringArray>
[      'ModernCRM', 'AnalyticsEngine',        'ModernHR',   'BillingSystem',
   'ThirdPartyAPI',       'LegacyCRM']
Length: 6, dtype: str

In [5]:
df.target_label.unique()

<StringArray>
[        'HTTP Status',      'Critical Error',      'Security Alert',
               'Error', 'System Notification',      'Resource Usage',
         'User Action',      'Workflow Error', 'Deprecation Warning']
Length: 9, dtype: str

In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN

# 1. load your data
# csv_path = r"synthetic_logs.csv"
# df = pd.read_csv(csv_path, parse_dates=[0], keep_default_na=False)

# make sure the column exists and is text
messages = df['log_message'].astype(str).tolist()

# 2. embed with a sentence transformer
model = SentenceTransformer('all-MiniLM-L6-v2')  # lightweight & good for semantic similarity
embeddings = model.encode(messages, show_progress_bar=True, convert_to_numpy=True)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/76 [00:00<?, ?it/s]

In [7]:
embeddings[:2]

array([[-1.02939650e-01,  3.35458964e-02, -2.20260918e-02,
         1.55101146e-03, -9.86918062e-03, -1.78956270e-01,
        -6.34409934e-02, -6.01761565e-02,  2.81109177e-02,
         5.99619336e-02, -1.72617678e-02,  1.43369730e-03,
        -1.49560049e-01,  3.15286336e-03, -5.66030741e-02,
         2.71685310e-02, -1.49890324e-02, -3.54037695e-02,
        -3.62936556e-02, -1.45410830e-02, -5.61489770e-03,
         8.75538513e-02,  4.55120876e-02,  2.50964146e-02,
         1.00187995e-02,  1.24266678e-02, -1.39923573e-01,
         7.68696368e-02,  3.14095467e-02, -4.15248377e-03,
         4.36902568e-02,  1.71250198e-02, -8.00950527e-02,
         5.74005954e-02,  1.89091992e-02,  8.55262130e-02,
         3.96399386e-02, -1.34371832e-01, -1.44358294e-03,
         3.06704384e-03,  1.76854014e-01,  4.44887392e-03,
        -1.69274751e-02,  2.24267188e-02, -4.35049385e-02,
         6.09029364e-03, -9.98165552e-03, -6.23972826e-02,
         1.07372459e-02, -6.04896760e-03, -7.14660957e-0

In [8]:
# 3. cluster with DBSCAN
# epsilon and min_samples will need tuning depending on dataset/closeness of messages.
# metric='cosine' works well for transformer vectors.
clustering = DBSCAN(eps=0.2, min_samples=1, metric='cosine')
labels = clustering.fit_predict(embeddings)

# 4. attach labels back to data frame
df['cluster'] = labels

# inspect some clusters
# for cluster_id in sorted(df['cluster'].unique()):
#     print(f"Cluster {cluster_id} (n={len(df[df['cluster'] == cluster_id])}):")
#     for msg in df[df['cluster'] == cluster_id]['log_message'].head(5):
#         print("    ", msg)
#     print()

In [9]:
df['cluster'].head(5)

0    0
1    1
2    2
3    0
4    0
Name: cluster, dtype: int64

In [10]:
# count records per cluster, keep only those with more than 10 entries
cluster_counts = df['cluster'].value_counts()
big_clusters = cluster_counts[cluster_counts > 10].sort_values(ascending=False)

for cid, cnt in big_clusters.items():
    print(f"Cluster {cid} (n={cnt}):")
    # show 5 messages from this cluster
    msgs = df.loc[df['cluster'] == cid, 'log_message'].head(5)
    for m in msgs:
        print("   ", m)
    print()

Cluster 0 (n=1017):
    nova.osapi_compute.wsgi.server [req-b9718cd8-f65e-49cc-8349-6cf7122af137 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" status: 200 len: 1893 time: 0.2675118
    nova.osapi_compute.wsgi.server [req-4895c258-b2f8-488f-a2a3-4fae63982e48 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" HTTP status code -  200 len: 211 time: 0.0968180
    nova.osapi_compute.wsgi.server [req-ee8bc8ba-9265-4280-9215-dbe000a41209 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" RCODE  200 len: 1874 time: 0.2280791
    nova.osapi_compute.wsgi.server [req-f0bffbc3-5ab0-4916-91c1-0a61dd7d4ec2 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54

In [11]:
import re

def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (out|in).": "User Action",
        r"Backup (started|ended) at.*": "System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version  .*" : "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"system reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }
    for pattern, label in regex_patterns.items():
        if re.match(pattern, log_message, flags=re.IGNORECASE):
            return label
    return None

In [12]:
classify_with_regex("User User123 Logged Out.")

'User Action'

In [13]:
df['regex_label'] = df['log_message'].apply(classify_with_regex)
df[df.regex_label.notnull()]

,timestamp,source,log_message,target_label,cluster,regex_label
7,10-11-2025 08:44,ModernHR,File data_6169.csv uploaded successfully by us...,System Notification,4,System Notification
14,01-04-2025 01:43,ThirdPartyAPI,File data_3847.csv uploaded successfully by us...,System Notification,4,System Notification
15,05-01-2025 09:41,ModernCRM,Backup completed successfully.,System Notification,8,System Notification
18,2/22/2025 17:49,ModernCRM,Account with ID 5351 created by User634.,User Action,9,User Action
27,9/24/2025 19:57,ThirdPartyAPI,User User685 logged out.,User Action,11,User Action
...,...,...,...,...,...,...
2368,12/13/2025 20:04,ThirdPartyAPI,Disk cleanup completed successfully.,System Notification,32,System Notification
2381,09-05-2025 06:39,ThirdPartyAPI,Disk cleanup completed successfully.,System Notification,32,System Notification
2394,04-03-2025 13:13,ModernHR,Disk cleanup completed successfully.,System Notification,32,System Notification
2395,05-02-2025 14:29,ThirdPartyAPI,Backup ended at 2025-05-06 11:23:16.,System Notification,13,System Notification


In [14]:
df.shape

(2410, 6)

In [15]:
df_non_regex = df[df['regex_label'].isnull()].copy()
df_non_regex.shape

(1968, 6)

In [16]:
print(df_non_regex['target_label'].value_counts()[df_non_regex['target_label'].value_counts() <= 5].index.tolist())

['Workflow Error', 'Deprecation Warning']


In [17]:
df_non_legacy = df_non_regex[df_non_regex.source != 'LegacyCRM']
df_non_legacy.source.unique()

<StringArray>
['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem', 'ThirdPartyAPI']
Length: 5, dtype: str

In [18]:
filtered_messages = df_non_legacy['log_message'].astype(str).tolist()

model = SentenceTransformer('all-MiniLM-L6-v2')  # lightweight & good for semantic similarity
filtered_embeddings = model.encode(filtered_messages, show_progress_bar=True, convert_to_numpy=True)
filtered_embeddings[:2]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/62 [00:00<?, ?it/s]

array([[-1.02939650e-01,  3.35458964e-02, -2.20260918e-02,
         1.55101146e-03, -9.86918062e-03, -1.78956270e-01,
        -6.34409934e-02, -6.01761565e-02,  2.81109177e-02,
         5.99619336e-02, -1.72617678e-02,  1.43369730e-03,
        -1.49560049e-01,  3.15286336e-03, -5.66030741e-02,
         2.71685310e-02, -1.49890324e-02, -3.54037695e-02,
        -3.62936556e-02, -1.45410830e-02, -5.61489770e-03,
         8.75538513e-02,  4.55120876e-02,  2.50964146e-02,
         1.00187995e-02,  1.24266678e-02, -1.39923573e-01,
         7.68696368e-02,  3.14095467e-02, -4.15248377e-03,
         4.36902568e-02,  1.71250198e-02, -8.00950527e-02,
         5.74005954e-02,  1.89091992e-02,  8.55262130e-02,
         3.96399386e-02, -1.34371832e-01, -1.44358294e-03,
         3.06704384e-03,  1.76854014e-01,  4.44887392e-03,
        -1.69274751e-02,  2.24267188e-02, -4.35049385e-02,
         6.09029364e-03, -9.98165552e-03, -6.23972826e-02,
         1.07372459e-02, -6.04896760e-03, -7.14660957e-0

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Get target labels for filtered data
y_filtered = df_non_legacy['target_label'].values

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    filtered_embeddings, y_filtered, test_size=0.2, random_state=42
)

# Train logistic regression model
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Evaluate
train_score = lr_model.score(X_train, y_train)
test_score = lr_model.score(X_test, y_test)

print(f"Train accuracy: {train_score:.4f}")
print(f"Test accuracy: {test_score:.4f}")

Train accuracy: 0.9955
Test accuracy: 0.9924


In [20]:
import joblib

joblib.dump(lr_model, '../models/log_classifier.joblib')

['../models/log_classifier.joblib']